# Laboratorio 3: Modelos de Regresión Lineal
## Consultoría para SmartStay Advisors

### Integrantes: 
- ***22213 Jorge Chupina*** 
- ***241452 Yehosua Hércules***
- ***231270 Gadiel Ocaña***

En esta primera fase de la consultoría, se realiza un análisis exploratorio detallado del conjunto de datos proporcionado. El objetivo principal es predecir el precio de las propiedades, por lo que nos enfocaremos en entender la distribución de los datos, realizar agrupamiento y evaluar la relación de las variables independientes con nuestra variable respuesta.

In [ ]:
# Instalar librerias necesarias

%pip install pandas numpy matplotlib seaborn pyreadr scikit-learn scipy

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadr
from sklearn.cluster import KMeans
import scipy.stats as stats

# Configuración para las gráficas
sns.set_theme(style="whitegrid")

# Cargamos el conjunto de datos original
resultado = pyreadr.read_r('listings.RData')
nombre_objeto = list(resultado.keys())[0]
df = resultado[nombre_objeto]

# Verificamos las dimensiones iniciales
print(f"El dataset contiene {df.shape[0]} filas y {df.shape[1]} columnas.")

El dataset contiene 171748 filas y 80 columnas.


## 1. Descripción de las Variables Seleccionadas
Dado que el dataset original cuenta con 80 columnas, hemos seleccionado las más relevantes para nuestro objetivo de predecir el precio y la ocupación. Las variables se agrupan de la siguiente manera:

**Variable Respuesta**

- price: Es el precio por noche de la propiedad. Viene en formato de texto con símbolos de moneda, por lo que requerirá limpieza para usarla en los modelos.

**Variables de la Propiedad**

- property_type y room_type: Categorizan el tipo de inmueble y el nivel de privacidad (ej. casa entera, habitación privada).

- accommodates: Capacidad máxima de huéspedes.

- bedrooms y beds: Cantidad de habitaciones y camas.

- bathrooms / bathrooms_text: Cantidad de baños disponibles.

**Variables de Ubicación**

- city y neighbourhood: Nos indican la ubicación textual de la propiedad.

- latitude y longitude: Coordenadas exactas, fundamentales para nuestro análisis de grupos (clustering) espacial.

**Variables de Reputación y Desempeño**

- number_of_reviews: Total de evaluaciones recibidas.

- review_scores_rating: Calificación promedio de los huéspedes.

- availability_365: Días disponibles en el año.

In [5]:
# Verificamos si los datos son solo de una ciudad o de varias
ciudades_unicas = df['city'].value_counts()
print("Distribución de propiedades por ciudad:")
print(ciudades_unicas.head(10)) 
# Mostramos el top 10

Distribución de propiedades por ciudad:
city
Los Angeles, California      45585
New York, New York           36261
Hawaii                       33457
San Diego, California        13162
Austin, Texas                10533
Chicago, Illinois             8660
San Francisco, California     7535
Washington, D.C.              6374
Rhode Island                  5762
Boston, Massachusetts         4419
Name: count, dtype: int64


## 2. Preprocesamiento y Limpieza
Como se notó en la exploración inicial, la columna "price" necesita limpieza ya que viene como texto, incluyendo signos de dólar y comas. Además, debemos descartar los registros que no tengan un precio definido, ya que no nos servirán para entrenar nuestro modelo.

In [6]:
# Limpieza price (quita $ y comas, y que sea numeric)
df['price'] = df['price'].astype(str).str.replace(r'[\$,]', '', regex=True).str.strip()
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df_clean = df.dropna(subset=['price']).copy()

# Eliminamos valores nulos en la variable respuesta
df_clean = df.dropna(subset=['price']).copy()

print(f"Tras limpiar los precios nulos, nos quedamos con {df_clean.shape[0]} registros útiles.")

Tras limpiar los precios nulos, nos quedamos con 76246 registros útiles.
